# M1M3 Faults vs Elevation Angle

Evaluate the number of M1M3 Faults for different elevation angles

Author: Bruno C. Quint
Associated tickets:
  - [SITCOM-2130](https://ls.st/SITCOM-2130)

In [ ]:
day_obs_start = 20250415
day_obs_end = 20250805

## Setup Notebook

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from lsst.sitcom.vandv.m1m3 import sitcom2130

## Analysis

The first plot shows the histogram of the number of faults versus the elevation angle for this period.  
The log-y is needed to show data with fewer faults.  
  
The second plot shows when the faults happened and the elevation angle at that time.  

In [ ]:
# Query all the faults inside the time range
faults_df = await sitcom2130.query_m1m3_faults(day_obs_start, day_obs_end)

M1M3 has lots of different types of faults.  
For the purpose of this analysis, we want to exclude the faults that are not triggered by normal operations.  
These are faults like: door open, lights on, no air, etc.  
The cell below selects the interlock faults 
 and print out unique options.

In [ ]:
interlock_faults_df = faults_df[faults_df.errorReport.str.contains("Interlock")]
print(interlock_faults_df.errorReport.unique())

Based on the output above, let's exclude the following errors:
  - Interlock Air Supply Off
  - Interlock Aux Power Bus Off
  - Interlock Cabinet Door Opened
  - Interlock lost GIS Heartbeat
  - Interlock Thermal Equipment Off

Let's keep "Interlock TMA Motion Stop" because this might come from a fault associated with M1M3. 

In [ ]:
exclude_errors = [
    "Interlock Air Supply Off",
    "Interlock Aux Power Bus Off",
    "Interlock Cabinet Door Opened",
    "Interlock lost GIS Heartbeat",
    "Interlock Thermal Equipment Off"  
]

# The ~ means "not"
filtered_faults_df = faults_df[~ faults_df.errorReport.str.contains("Interlock")]

In [ ]:
figure = sitcom2130.plot_faults_vs_elevation(filtered_faults_df)
figure.savefig(f"m1m3_faults_vs_tma_elevation_from_{day_obs_start}_to_{day_obs_end}.png")